1. 导入必要的库

In [ ]:
# 导入必要的库
import pandas as pd  # 用于读取CSV文件
import numpy as np  # 用于数组处理
from PIL import Image  # 用于处理图像
import matplotlib.pyplot as plt  # 用于图像显示
from mindspore import Tensor  # 用于Tensor操作
import mindspore
import mindspore.dataset as ds  # 用于数据加载
from mindspore.dataset import vision  # 对图像进行数据预处理
import mindspore.nn as nn  # 用于定义神经网络模块
from mindspore.common.initializer import TruncatedNormal  # 用于初始化网络权重


2.加载数据集并进行预处理
这段代码定义了一个自定义数据集 NewtonRingData，从CSV文件加载牛顿环图像及其半径标签，并进行图像预处理（归一化和裁剪）。使用 MindSpore 的 GeneratorDataset 创建数据加载器，并将图像数据格式从 HWC 转换为 CHW 格式。最后，数据加载器按批次（每次5个样本）提供处理后的图像和半径标签，供模型训练使用。

In [ ]:
# 定义自定义数据集类
class NewtonRingData():
    """自定义牛顿环数据集"""

    def __init__(self, csv_file):
        # 读取CSV文件，包含图像路径、坐标和半径信息
        self.data_frame = pd.read_csv(csv_file)

    def __len__(self):
        # 返回数据集的大小，即图像数量
        return len(self.data_frame)

    def __getitem__(self, idx):
        # 获取指定索引的数据
        img_name = self.data_frame.iloc[idx, 0]  # 图像路径
        image = Image.open(img_name)  # 使用PIL打开图像
        image = np.array(image, dtype=np.float32)  # 转换为numpy数组
        image = (1 / 255) * image  # 图像归一化到0-1之间

        # 裁剪图像为224x224的尺寸
        image = image[240 - 112:240 + 112, 320 - 112:320 + 112]  # 裁剪图像中心区域

        # 获取该图像对应的半径标签
        radius = self.data_frame.iloc[idx, -1].astype(np.float32)

        return image, radius

# 加载数据集并创建训练数据集
train_data = NewtonRingData(csv_file='./Ideal/train/data.csv')

# 使用MindSpore的GeneratorDataset加载数据
train_data_loader = ds.GeneratorDataset(source=train_data, column_names=['img', 'Radius'])
train_data_loader = train_data_loader.map(vision.HWC2CHW(), 'img')  # 图像数据格式转换

# 批处理数据，每批次包含5个样本
train_data_loader = train_data_loader.batch(5)


3. 定义VGG网络
这段代码定义了一个基于VGG结构的神经网络，用于预测曲率半径。首先，定义了一个卷积模块 vgg_block，包含多个卷积层和最大池化层。然后，将多个卷积块组合成特征提取部分。接着，使用全连接层构建分类器，用于输出预测的曲率半径。网络结构由四个卷积块和三个全连接层组成。最后，定义了 construct 方法，完成前向传播过程，从特征提取到分类输出。

In [ ]:
# 定义VGG网络的构建模块：vgg_block
def vgg_block(num_convs, in_channels, out_channels):
    """
    创建一个VGG的卷积块，由多个卷积层组成，后接最大池化层。
    """
    layers = []  # 存放层的列表
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, pad_mode='pad', weight_init=TruncatedNormal(0.02)))  # 卷积层
        layers.append(nn.ReLU())  # 激活函数ReLU
        in_channels = out_channels  # 下一层的输入通道数等于当前层的输出通道数

    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))  # 最大池化层，大小为2，步长为2
    return nn.SequentialCell(layers)  # 返回构建好的模块

# 定义完整的VGG网络
class VGG(nn.Cell):
    def __init__(self, num_classes=1):  # 仅预测曲率半径，因此num_classes设为1
        super(VGG, self).__init__()

        # 特征提取部分，由多个vgg_block组成
        self.features = nn.SequentialCell([
            vgg_block(2, 3, 16),   # 第一个块：2层卷积，输入3通道，输出16通道
            vgg_block(3, 16, 32),  # 第二个块：3层卷积，输入16通道，输出32通道
            vgg_block(3, 32, 64),  # 第三个块：3层卷积，输入32通道，输出64通道
            vgg_block(3, 64, 128)  # 第四个块：3层卷积，输入64通道，输出128通道
        ])

        # 分类部分，由全连接层构成
        self.classifier = nn.SequentialCell([
            nn.Dense(128 * 14 * 14, 4096, weight_init=TruncatedNormal(0.02)),  # 输入128*14*14，输出4096
            nn.ReLU(),  # 激活函数ReLU
            nn.Dense(4096, 4096, weight_init=TruncatedNormal(0.02)),  # 第二个全连接层，输出4096
            nn.ReLU(),  # 激活函数ReLU
            nn.Dense(4096, num_classes, weight_init=TruncatedNormal(0.02))  # 最后一个全连接层，输出半径预测结果
        ])

    def construct(self, x):
        """前向传播"""
        x = self.features(x)  # 特征提取部分
        x = x.view(x.shape[0], -1)  # 展平特征图，为全连接层做准备
        x = self.classifier(x)  # 分类部分
        return x


4. 训练模型
定义了一个训练模型的主函数 `train_model`，用于训练VGG网络。函数首先定义了损失函数（RMSE损失）和优化器（Adam）。在每一轮训练中，函数执行前向传播、计算损失，并通过反向传播更新模型参数。训练过程持续多个epoch，并输出每个epoch的平均损失。最后，返回每轮训练的损失历史。

In [ ]:
def train_model(model, dataset, epochs=50, learning_rate=1e-5):
    """
    模型训练主函数：定义损失函数、优化器、前向传播、反向传播并执行多轮训练。

    参数：
    - model: 要训练的模型
    - dataset: MindSpore 格式的数据加载器
    - epochs: 训练轮数（默认 50）
    - learning_rate: 学习率（默认 1e-5）

    返回：
    - losses: 每轮训练的平均损失列表
    """
    # 1. 定义损失函数和优化器
    loss_fn = nn.RMSELoss()
    optimizer = nn.Adam(model.trainable_params(), learning_rate=learning_rate)

    # 2. 定义前向传播函数
    def forward_fn(data, label):
        logits = model(data)  # 前向传播
        loss = loss_fn(logits, label)  # 只使用半径作为输出
        return loss, logits

    # 3. 自动求导函数（带辅助返回）
    grad_fn = mindspore.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

    # 4. 开始训练循环
    losses = []
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}\n-------------------------------")
        size = dataset.get_dataset_size()
        total_loss = 0
        model.set_train()

        for batch, (data, label) in enumerate(dataset.create_tuple_iterator()):
            (loss, _), grads = grad_fn(data, label)
            optimizer(grads)
            total_loss += loss

            if batch % 5 == 0:
                print(f"Loss: {loss.asnumpy():.6f}  [{batch:>3d}/{size:>3d}]")

        avg_loss = total_loss / size
        losses.append(avg_loss)
        print(f"Epoch {epoch + 1} completed. Avg Loss: {avg_loss}\n")

    print("Training Done!")
    return losses

# 创建VGG模型实例
vgg = VGG(num_classes=1)
# 训练模型
loss_history = train_model(vgg, train_data_loader, epochs=50, learning_rate=1e-5)



5. 保存与加载模型

In [ ]:
# 保存模型的checkpoint文件
mindspore.save_checkpoint(vgg, "model.ckpt")
print("Saved Model to model.ckpt")  # 输出提示，表示模型已保存为"model.ckpt"

# 加载checkpoint文件并将参数加载到模型中
param_dict = mindspore.load_checkpoint("model.ckpt")  # 加载存储的checkpoint文件
param_not_load, _ = mindspore.load_param_into_net(vgg, param_dict)  # 将加载的参数赋值到模型中


6. 预测步骤
这段代码实现了加载并处理测试图像，使用VGG模型预测图像中的曲率半径，并计算预测结果与真实值之间的相对误差。首先，它加载图像并进行归一化和裁剪，调整图像的维度以适配模型输入。然后，图像通过VGG模型进行预测，得到预测的半径值，接着将其转换为实际单位，并计算与真实半径之间的相对误差，最终输出预测结果和误差。

In [ ]:
# 预测步骤：加载图像并进行预测
img_name = './Ideal/test/1.bmp'  # 测试图像路径
img = Image.open(img_name)  # 打开图像文件
plt.imshow(img)  # 显示图像
plt.show()

# 图像处理：归一化并裁剪为224x224
img = np.array(img)
img = (1 / 255) * img  # 归一化
img = img[240 - 112:240 + 112, 320 - 112:320 + 112]  # 裁剪为224x224大小

# 转换维度并增加批次维度
img = np.transpose(img, (2, 0, 1))  # 转换为(C, H, W)
img = np.expand_dims(img, axis=0)  # 增加批次维度

# 转换为Tensor并预测
img = Tensor(img, mindspore.float32)
vgg.set_train(False)  # 设置为评估模式
output = vgg(img)

# 获取预测的半径
R_e = output[0][0]  # 假设输出的第一个值为半径估计值

R=0.85
# 将预测的半径转换为实际单位
R_ee = R_e
# 计算相对误差
relativ_error = abs(R_ee - R) / R * 100  # 计算相对误差百分比

# 输出结果
print(f'Predicted Radius: {R_ee}')  # 打印预测的半径
print(f'Relative Error: {relativ_error}%')  # 打印相对误差
